In [2]:
# Cell 1 — Import Libraries
import os
import cv2
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

output_path = "../output/final_dataset"
FINAL_CLASSES = ['Plastic', 'Glass', 'Metal', 'Paper', 'Organic', 'Non-Recyclable']
COLORS = {
    'Plastic':         (255, 0, 0),
    'Glass':           (0, 255, 0),
    'Metal':           (0, 0, 255),
    'Paper':           (255, 255, 0),
    'Organic':         (0, 255, 255),
    'Non-Recyclable':  (255, 0, 255),
}

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [3]:
# Cell 2 — Fast Check Every Image Has a Label
print("🔍 بنتحقق إن كل صورة عندها label...\n")

missing_labels = 0
empty_labels = 0
valid = 0

for split in ['train', 'val', 'test']:
    img_stems = set(Path(f).stem for f in glob.glob(f"{output_path}/{split}/images/*"))
    lbl_stems = set(Path(f).stem for f in glob.glob(f"{output_path}/{split}/labels/*"))
    
    # صور من غير labels
    missing = img_stems - lbl_stems
    missing_labels += len(missing)
    
    # صور عندها labels
    matched = img_stems & lbl_stems
    valid += len(matched)
    
    print(f"  {split}:")
    print(f"    صور:    {len(img_stems):,}")
    print(f"    labels: {len(lbl_stems):,}")
    print(f"    ناقص:   {len(missing):,}")

print(f"\n  ✅ صور عندها label: {valid:,}")
print(f"  ❌ صور من غير label: {missing_labels:,}")

🔍 بنتحقق إن كل صورة عندها label...

  train:
    صور:    424,279
    labels: 424,279
    ناقص:   0
  val:
    صور:    6,572
    labels: 6,572
    ناقص:   0
  test:
    صور:    6,475
    labels: 6,475
    ناقص:   0

  ✅ صور عندها label: 437,326
  ❌ صور من غير label: 0


In [1]:
# Cell 1 — Import Libraries (No cv2)
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

output_path = "../output/final_dataset"
FINAL_CLASSES = ['Plastic', 'Glass', 'Metal', 'Paper', 'Organic', 'Non-Recyclable']
COLORS = ['red', 'green', 'blue', 'yellow', 'cyan', 'magenta']

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# Cell 2 — Visualize Sample Annotations per Class
random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, cls_name in enumerate(FINAL_CLASSES):
    class_id = idx
    found = False

    labels = glob.glob(f"{output_path}/train/labels/*.txt")[:300]

    for label_path in labels:
        with open(label_path, 'r') as f:
            lines = f.readlines()

        has_class = any(
            line.strip() and int(float(line.split()[0])) == class_id
            for line in lines if line.strip()
        )
        if not has_class:
            continue

        stem = Path(label_path).stem
        img_path = f"{output_path}/train/images/{stem}.jpg"
        if not os.path.exists(img_path):
            img_path = f"{output_path}/train/images/{stem}.png"
        if not os.path.exists(img_path):
            continue

        try:
            img = Image.open(img_path).convert("RGB")
        except:
            continue

        w, h = img.size
        ax = axes[idx]
        ax.imshow(img)

        for line in lines:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cid = int(float(parts[0]))
            x_c, y_c, bw, bh = map(float, parts[1:5])
            x1 = (x_c - bw/2) * w
            y1 = (y_c - bh/2) * h
            rect = patches.Rectangle(
                (x1, y1), bw*w, bh*h,
                linewidth=2,
                edgecolor=COLORS[cid],
                facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1-5, FINAL_CLASSES[cid],
                   color=COLORS[cid], fontsize=8, fontweight='bold')

        ax.set_title(cls_name, fontsize=12)
        ax.axis('off')
        img.close()
        found = True
        break

    if not found:
        axes[idx].set_title(f"{cls_name} - مش لاقي")
        axes[idx].axis('off')

plt.suptitle('Sample Annotations per Class', fontsize=15)
plt.tight_layout()
plt.savefig('../notebooks/annotation_samples.png', dpi=100)
plt.show()
print("✅ خلص!")

: 